# DiffuGroundingDINO trên Kaggle 2×T4

Finetune `groundingdino_swint_ogc.pth` với nhánh diffusion trên reference point (xem `object-detection/diffu_grounding_dino/README.md`). Notebook này dùng **cả 2 GPU T4** qua `torchrun` để tăng tốc, khác với `tools/run_train.py` (dùng cho GPU server riêng, chỉ 1 GPU mỗi lần).

**Trước khi chạy:**
1. Settings > Accelerator: **GPU T4 x2**. Settings > Internet: **On** (để `git clone`).
2. Add Data — 2 Kaggle Dataset đã chuẩn bị sẵn từ máy dev (zip trong `object-detection/weights/` và `object-detection/data/`):
   - `diffu-gdino-weights` chứa `diffu_grounding_dino_weights.zip` (groundingdino_swint_ogc.pth + bert-base-uncased, ~1GB)
   - `diffu-gdino-coco-data` chứa `data_coco_for_diffu_gdino.zip` (coco + coco_minitrain, ~4,6GB)
3. Code lấy trực tiếp từ GitHub (`git clone`) chứ không cần đóng gói riêng — sửa `REPO_URL`/`BRANCH` ở cell dưới nếu khác.

In [ ]:
import os

REPO_URL = "https://github.com/CryAndRRich/object-detection.git"
BRANCH = "main"

USE_DIFFUSION = True
EPOCHS = 15
BATCH_SIZE = 4          # mỗi GPU -- torchrun chạy 2 tiến trình, tổng batch = 2x giá trị này
LR = 1e-4
SAMPLING_STEPS = 3      # số bước DDIM lúc eval
NUM_WORKERS = 2
RUN_TESTS = True
RESUME = ""            # đường dẫn checkpoint nếu resume phiên trước, để trống nếu train từ đầu

WORK = "/kaggle/working"
REPO = f"{WORK}/object-detection"
CODE = f"{REPO}/diffu_grounding_dino"

# Kaggle Dataset input -- đổi tên cho khớp dataset bạn đã add
WEIGHTS_ZIP_INPUT = "/kaggle/input/diffu-gdino-weights/diffu_grounding_dino_weights.zip"
DATA_ZIP_INPUT = "/kaggle/input/diffu-gdino-coco-data/data_coco_for_diffu_gdino.zip"

OUT_DIR = f"{WORK}/output/diffu_run1"

## 1. Kiểm tra môi trường + cài lib

torch/torchvision Kaggle đã có sẵn khớp CUDA của máy -- **không cài lại**, chỉ thêm 3 lib nhẹ project cần (đúng `requirements.txt`, trừ torch/torchvision).

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| GPU count:", torch.cuda.device_count())
assert torch.cuda.device_count() == 2, (
    f"cần 2 GPU T4 (Settings > Accelerator > GPU T4 x2), đang thấy {torch.cuda.device_count()}"
)

!pip install -q 'transformers>=4.30,<5' 'scipy>=1.9' 'pycocotools>=2.0.6'

## 2. Lấy code từ GitHub

In [ ]:
import subprocess

if not os.path.isdir(REPO):
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, REPO],
        check=True,
    )
os.chdir(CODE)
print("cwd:", os.getcwd())

## 3. Giải nén weights + data từ Kaggle Dataset input

In [ ]:
import zipfile

os.makedirs(f"{REPO}/weights", exist_ok=True)
with zipfile.ZipFile(WEIGHTS_ZIP_INPUT) as zf:
    zf.extractall(f"{REPO}/weights")

PRETRAIN = f"{REPO}/weights/diffu_grounding_dino/groundingdino_swint_ogc.pth"
BERT_DIR = f"{REPO}/weights/diffu_grounding_dino/bert-base-uncased"
assert os.path.exists(PRETRAIN), PRETRAIN
assert os.path.isdir(BERT_DIR), BERT_DIR
print("weights OK:", PRETRAIN, BERT_DIR)

In [ ]:
os.makedirs(f"{REPO}/data", exist_ok=True)
with zipfile.ZipFile(DATA_ZIP_INPUT) as zf:
    zf.extractall(f"{REPO}/data")

TRAIN_ANN = f"{REPO}/data/coco_minitrain/annotations/instances_minitrain2017.json"
TRAIN_IMAGES = f"{REPO}/data/coco_minitrain/images/train2017"
VAL_ANN = f"{REPO}/data/coco/annotations/instances_val2017.json"
VAL_IMAGES = f"{REPO}/data/coco/val2017"
for p in (TRAIN_ANN, TRAIN_IMAGES, VAL_ANN, VAL_IMAGES):
    assert os.path.exists(p), p
print("data OK")

## 4. Convert COCO-minitrain sang ODVG + dựng datasets json

In [ ]:
ANN_DIR = f"{WORK}/annotations"
os.makedirs(ANN_DIR, exist_ok=True)
ODVG_JSONL = f"{ANN_DIR}/minitrain_odvg.jsonl"
LABEL_MAP = f"{ANN_DIR}/label_map.json"

!python tools/coco2odvg.py --input {TRAIN_ANN} --output-jsonl {ODVG_JSONL} --output-label-map {LABEL_MAP}

In [ ]:
import json

DATASETS_JSON = f"{WORK}/datasets.json"
datasets_cfg = {
    "train": [{"root": TRAIN_IMAGES, "anno": ODVG_JSONL, "label_map": LABEL_MAP, "dataset_mode": "odvg"}],
    "val": [{"root": VAL_IMAGES, "anno": VAL_ANN, "dataset_mode": "coco"}],
}
with open(DATASETS_JSON, "w") as f:
    json.dump(datasets_cfg, f)

CONFIG = "config/cfg_odvg_diffusion.py" if USE_DIFFUSION else "config/cfg_odvg.py"
OPTIONS = [
    f"text_encoder_type={BERT_DIR!r}",
    f"epochs={EPOCHS}",
    f"batch_size={BATCH_SIZE}",
    f"lr={LR}",
    f"num_workers={NUM_WORKERS}",
]
if USE_DIFFUSION:
    OPTIONS.append(f"diff_sampling_timesteps={SAMPLING_STEPS}")
opt_str = " ".join(OPTIONS)
print(DATASETS_JSON, "\n", CONFIG, "\n", OPTIONS)

## 5. Verification (80 test CPU + checkpoint key-compat)

Chạy trước khi train thật -- rẻ, bắt được lỗi wiring trước khi tốn GPU-hour. Xem README mục "Verification" để biết 8 bước tương ứng.

In [ ]:
if RUN_TESTS:
    !python tests/run_all.py

In [ ]:
!python tools/check_checkpoint.py -c {CONFIG} --checkpoint {PRETRAIN} --options {opt_str}

## 6. Train — **2 GPU song song qua `torchrun`**

`--nproc_per_node=2` chạy 2 tiến trình, mỗi tiến trình 1 GPU, `DistributedDataParallel` tự đồng bộ gradient sau mỗi bước (đã verify cơ chế này đúng bằng `tests/test_ddp.py` trước khi đưa lên đây). `BATCH_SIZE` ở trên là **mỗi GPU** -- tổng batch thực tế train là `2 * BATCH_SIZE`.

In [ ]:
n_gpu = torch.cuda.device_count()
assert n_gpu == 2, f"notebook này thiết kế cho 2 GPU, đang thấy {n_gpu}"

if RESUME:
    launch_args = f"--resume {RESUME}"
else:
    launch_args = f"--pretrain_model_path {PRETRAIN} --finetune_ignore time_ diffusion"

cmd = (
    f"torchrun --standalone --nproc_per_node={n_gpu} main.py "
    f"-c {CONFIG} --datasets {DATASETS_JSON} --output_dir {OUT_DIR} "
    f"{launch_args} --options {opt_str}"
)
print(cmd)
!{cmd}

## 7. Đọc log

In [ ]:
log_path = f"{OUT_DIR}/log.txt"
if os.path.exists(log_path):
    with open(log_path) as f:
        for line in f:
            print(line.rstrip())
else:
    print("chưa có log.txt ở", log_path)

## 8. Eval, quét số bước sampling

Cũng chạy 2 GPU cho nhanh, dù eval không throughput-critical bằng train.

In [ ]:
for s in (1, 3, 5, 10):
    eval_out = f"{WORK}/output/eval_s{s}"
    cmd = (
        f"torchrun --standalone --nproc_per_node={n_gpu} main.py "
        f"-c {CONFIG} --datasets {DATASETS_JSON} --output_dir {eval_out} --eval "
        f"--resume {OUT_DIR}/checkpoint_best_regular.pth "
        f"--options {opt_str} diff_sampling_timesteps={s}"
    )
    print(cmd)
    !{cmd}

## 9. Dọn dẹp + nhắc báo cáo

In [ ]:
!du -sh {OUT_DIR}
!ls -la {OUT_DIR}
print(f"Báo cáo kết quả kèm: EPOCHS={EPOCHS}, BATCH_SIZE/GPU={BATCH_SIZE}, n_gpu={n_gpu}, "
      f"tổng batch={BATCH_SIZE * n_gpu} (quy ước CLAUDE.md: luôn ghi iteration + batch size).")